# Practice Session 04: Networks from text

<font size="+2" color="blue">Additional results: account types</font>

Author: <font color="blue">Luca Franceschi</font>

E-mail: <font color="blue">luca.franceschi01@estudiant.upf.edu</font>

Date: <font color="blue">Due Oct. 12th, 18:30</font>

# 1. Create the directed mention network

In [ ]:
import io
import json
import gzip
import csv
import re
import itertools

from IPython.display import Image

In [ ]:
# Leave this code as-is

# Input file
COMPRESSED_INPUT_FILENAME = "CovidLockdownCatalonia.json.gz"

# These are the output files, leave as-is
OUTPUT_ALL_EDGES_FILENAME = "CovidLockdownCatalonia.csv"
OUTPUT_FILTERED_EDGES_FILENAME = "CovidLockdownCatalonia-min-weight-filtered.csv"
OUTPUT_CO_MENTIONS_FILENAME = "CovidLockdownCatalonia-co-mentions.csv"

## 1.1. Extract mentions

In [ ]:
# Leave this code as-is

def extract_mentions(text):
    return re.findall("@([a-zA-Z0-9_]{5,20})", text)

print(extract_mentions("RT @DiariDeSabadell: check this post by @EspaiNaturaSbd"))

## 1.2. Count mentions

In [ ]:
# Read the compressed input file and create the mentions_counter dictionary

mentions_counter = {}

with gzip.open(COMPRESSED_INPUT_FILENAME, "rt", encoding="utf-8") as input_file:
	for line in input_file:
		tweet = json.loads(line)
		author = tweet["user"]["screen_name"]
		message = tweet["full_text"]
		for mention in extract_mentions(message):
			key = (author, mention)
			if key in mentions_counter:
				mentions_counter[key] += 1
			else:
				mentions_counter[key] = 1

accountA = 'BCN_Mobilitat'
accountB = 'TMBinfo'
print('The account @%s mentions @%s %d times' % (accountA, accountB, mentions_counter.get((accountA, accountB))))

In [ ]:
# Print all the pairs of accounts (u,v) in which account u mentioned account v, and account v mentioned account u.

for (u, v) in mentions_counter:
	if (v, u) in mentions_counter and u<v:
		print('Accounts @%s and @%s mention each other' % (u, v))

In [ ]:
# Leave this code as-is

lines_written = 0
with io.open(OUTPUT_ALL_EDGES_FILENAME, "w") as output_file:
	writer = csv.writer(output_file, delimiter='\t', quotechar='"', lineterminator='\n')
	writer.writerow(["Source", "Target", "Weight"])
	for key in mentions_counter:
		author = key[0]
		mention = key[1]
		weight = mentions_counter[key]
		writer.writerow([author, mention, weight])
		lines_written += 1
		
print("Wrote %d lines to file %s" % (lines_written, OUTPUT_ALL_EDGES_FILENAME))

In [ ]:
# Create a file containing all (author, mention) pairs with a value greater or equal than 2

lines_written = 0
with io.open(OUTPUT_FILTERED_EDGES_FILENAME, "w") as output_file:
	writer = csv.writer(output_file, delimiter='\t', quotechar='"', lineterminator='\n')
	writer.writerow(["Source", "Target", "Weight"])
	for key in mentions_counter:
		weight = mentions_counter[key]
		if weight >= 2:
			author = key[0]
			mention = key[1]
			writer.writerow([author, mention, weight])
			lines_written += 1

print("Wrote %d lines to file %s" % (lines_written, OUTPUT_FILTERED_EDGES_FILENAME))

# 2. Visualize the directed mention network

## 2.1. Visualize the largest connected component


<font size="+1" color="red">What is the size of the largest connected component, both as a number of nodes and as a percentage of the nodes in the graph? What is the diameter of the largest connected component, disregarding edge direction? </font>

The size of the largest connected component is 699 nodes. That means that in a graph of 1600 nodes the largest connected component comprises the 43.69% of the graph.
The diameter of the largest connected component disregarding edge direction is 20.

In [ ]:
# Adjust width/height as needed

Image(url="mentions-largest-cc.png", width=1200)

<font size="+1" color="red">Replace this cell with some observations about this graph. Which accounts are mentioned by many other accounts? Why do you think these accounts are often mentioned? Which accounts mention many other accounts? Why do you think they do that?</font>

We can see that the nodes that have the most in-degree are public entities such as the "emergenciescat" or "govern" accounts. Also public representants such as the president of the Generalitat de Catalunya "QuimTorraiPla" and the Spanish Government "sanchezcastejon". This is because many people sent them mentions probably because of the lockdown or asking about Covid-19-related information. All the attention in social networks was directed to those public entity accounts because it was the main way of trying to talk with some public representant.

We can see that there are some accounts that mention many other ones (e.g.: enricgari, joanmariapique, AlbertRaurell or emocionycambio). This might be due to excess of free time during the lockdown and those people were probably complaining about some procedures that they disliked, which were promoted by those public entities.

## 2.2. Cluster the largest connected component


<font size="+1" color="red">Look at the cluster containing the account ``@salutcat``. Are there other thematically-related accounts in the same cluster? Name three of them. Indicate why do you think they are in the same cluster.</font>

Yes, there are other thematically-related accounts in the same cluster such as "emergenciescat", "gencat" or "mossos". That might be due to that they have the most in-degree ss we saw in the 2.1 section. That is probably due to the fact that those are the main accounts for public entities on Twitter (now X).

## 2.3. Examine degree distributions

In [ ]:
# Adjust width/height as needed

display(Image(url="mentions-largest-cc-indegree.png", width=400))

display(Image(url="mentions-largest-cc-outdegree.png", width=400))

<font size="+1" color="red">Replace this cell by a brief commentary, in your own words, about these degree distributions</font>

In both cases the in-degree and the out-degree distributions seem to be exponential (with base in the range (0, 1)). That is mainly because of the concept of homophily: social graphs are not even, accounts in the center of the graph are much more connected to others than nodes near the circumference (periphery) of the graph and accumulate most of the in-degree and the out-degree in very few central nodes.

# 3. Create the undirected co-mention network

In [ ]:
# Create the co_mentions_counter

co_mentions_counter = {}

with gzip.open(COMPRESSED_INPUT_FILENAME, "rt", encoding="utf-8") as input_file:
	for line in input_file:
		tweet = json.loads(line)
		author = tweet["user"]["screen_name"]
		message = tweet["full_text"]
		mentions = extract_mentions(message)
		for mention1 in mentions:
			for mention2 in mentions:
				if mention1 < mention2:
					key = (mention1, mention2)
					if key in co_mentions_counter:
						co_mentions_counter[key] += 1
					else:
						co_mentions_counter[key] = 1

In [ ]:
# KEEP AS-IS

print(co_mentions_counter[('emergenciescat', 'govern')])

In [ ]:
# Print all pairs of accounts that have been co-mentioned 20 times or more

for key in co_mentions_counter:
	if co_mentions_counter[key] >= 20:
		print(key)

In [ ]:
# Create the co-mentions file

with io.open(OUTPUT_CO_MENTIONS_FILENAME, "w") as output_file:
	writer = csv.writer(output_file, delimiter='\t', quotechar='"', lineterminator='\n')
	writer.writerow(["Source", "Target", "Weight"])
	for key in co_mentions_counter:
		author = key[0]
		mention = key[1]
		weight = co_mentions_counter[key]
		writer.writerow([author, mention, weight])

# 4. Visualize the undirected co-mention network in Cytoscape


In [ ]:
# Adjust width/height as needed

Image(url="co-mentions-min-degree-15.png", width=1200)

<font size="+1" color="red">Find one dense community in which nodes seem to be thematically related, for instance, a densely connected sub-graph or connected component in which (according to the names of the accounts) many nodes seem to have something in common. Replace this cell with a commentary on these two communities, indicating some example nodes and why do you think they are related.</font>

In a certain community (the one containing 'royo1_royo' and marmots') we can see that there are 50 nodes that are very densely related (in fact, they form a clique since they are connected by 1225 edges). That community is connected to another one (the one containing 'QuimTorraiPla', 'govern' and 'gencat') that is also very densely related (around 28 nodes). These two separate dense communities are connected through very few nodes (e.g.: 'saribes', 'Sedi1714').

The both communities seem to be almost fully catalan-independent accounts or people involved in politics. 

# Extra section

In [ ]:
extra_mentions_counter = {}

with gzip.open(COMPRESSED_INPUT_FILENAME, "rt", encoding="utf-8") as input_file:
	for line in input_file:
		tweet = json.loads(line)
		message = tweet["full_text"]
		mentions = extract_mentions(message)
		for account in mentions:
			if account in extra_mentions_counter:
				extra_mentions_counter[account] += 1
			else:
				extra_mentions_counter[account] = 1

# The list of 50 most mentioned accounts comes from the code below, and the types are manually inserted

# sorted_accounts = sorted(extra_mentions_counter.items(), key=lambda x:x[1], reverse=True)
# i=0
# while i<50:
# 	print(sorted_accounts[i][0])
# 	i += 1

accounts_types = {
	'emergenciescat': 'government institution',
	'QuimTorraiPla': 'politician',
	'sanchezcastejon': 'politician',
	'govern': 'government institution',
	'elnacionalcat': 'media',
	'salutcat': 'health-related',
	'XSalaimartin': 'journalist',
	'mossos': 'government institution',
	'FonsiLoaiza': 'journalist',
	'gmaemejota': 'individual',
	'VilaWeb': 'media',
	'gencat': 'government institution',
	'ChinaEmbEsp': 'government institution',
	'pablom_m': 'individual',
	'aleixsarri': 'politician',
	'324cat': 'media',
	'NicholsUprising': 'journalist',
	'eldiarioes': 'media',
	'vpartal': 'journalist',
	'ramirp': 'individual',
	'ReadToriko': 'individual',
	'joseantich': 'journalist',
	'iescolar': 'journalist',
	'_supergomez': 'individual',
	'luisgonzaloseg': 'journalist',
	'naciodigital': 'media',
	'Antoni_Gelonch': 'individual',
	'_theghettomonk': 'individual',
	'EnfrmraSaturada': 'journalist',
	'joethebrew': 'individual',
	'yeyaboya': 'journalist',
	'valemercurii': 'individual',
	'gerardotc': 'journalist',
	'CarlesHeredia': 'journalist',
	'publico_es': 'media',
	'pmarsupia': 'journalist',
	'tjparfitt': 'journalist',
	'SoniaVivasRive3': 'journalist',
	'_espatricia': 'journalist',
	'QHaRi': 'individual',
	'boye_g': 'journalist',
	'semgencat': 'health-related',
	'el_pais': 'media',
	'Alvisepf': 'individual',
	'juansotoivars': 'journalist',
	'Oriol_Pintos': 'individual',
	'btvnoticies': 'media',
	'RaholaOficial': 'journalist',
	'jmangues': 'individual',
	'G_Pisarello': 'politician'
}

with io.open("account-type.csv", "w") as output_file:
	writer = csv.writer(output_file, delimiter='\t', quotechar='"', lineterminator='\n')
	writer.writerow(["Source", "Type"])
	for key in accounts_types:
		account = key
		type = accounts_types[key]
		writer.writerow([account, type])


In [ ]:
Image(url="network-account-types.png", width=1200)

In [ ]:
Image(url="legend-account-types.gif", width=1200)

<font size="+1" color="red">COMMENTS ON THIS!!</font>


# DELIVER (individually)

Deliver a zip file containing:

* Your code as a Python notebook (a `.ipynb` file).
   * Remove all unnecessary elements
   * Add comments when needed
* Any png files that you inserted in the notebook

## Extra points available

For more learning and extra points, create a file `account-type.csv` containing the type of account of the top 50 accounts with the most mentions. You can use types "journalist", "media", "politician", "government institution", "individual", "health-related", etc. which you should categorize manually. Create a visualization of the **mentions** graph either including only these 50 accounts, or including more accounts but highlighting these top 50 with colors. Use broad categories as needed and **do not worry if there are some ambiguities in the categorization,** e.g., if you are not 100% sure on whether someone should be in one category or another; just do your best.

**Note:** if you go for the extra points, add ``<font size="+2" color="blue">Additional results: account types</font>`` at the top of your notebook.

<font size="-1" color="gray">(Remove this cell when delivering.)</font>


<font size="+2" color="#003300">I hereby declare that, except for the code provided by the course instructors, all of my code, text, and figures were produced by myself.</font>
